# LIDC-IDRI Nodule Segmentation — DPLinkNet50 + Attention

Runs the `lidcseg` package. Everything is configured by YAML in `configs/`;
this notebook only clones, locates the data, and launches runs.

**Before you start**
1. *Add data* → search `lidcidri` → add **zhangweiled/lidcidri**.
2. *Settings* → Accelerator → **GPU T4 x2** (or P100).
3. *Settings* → Internet → **On** (needed for the clone and ImageNet weights).

**What differs from the original notebook** — read `README.md` for the full
list, but the one that changes the numbers most: splits are grouped by
**patient**, so no patient's slices appear in both train and test. The
original split over slices, which put near-duplicate neighbouring slices on
both sides. Expect lower but trustworthy Dice.

In [ ]:
# ── 1. Get the code ───────────────────────────────────────────────────────
import os, sys, subprocess, pathlib

REPO = "https://github.com/heller007/LIDC_nodule_segmentation.git"
WORK = pathlib.Path("/kaggle/working/LIDC_nodule_segmentation")

if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(WORK)], check=True)
else:
    subprocess.run(["git", "-C", str(WORK), "pull", "--ff-only"], check=False)

os.chdir(WORK)
sys.path.insert(0, str(WORK / "src"))
print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── 2. Find the mounted dataset ───────────────────────────────────────────
# Kaggle's mount name varies; resolve it instead of hard-coding a path.
from lidcseg.data.index import find_dataset_root
import pathlib

candidates = sorted(pathlib.Path("/kaggle/input").glob("*"))
print("mounted:", [c.name for c in candidates])

DATA_ROOT = None
for c in candidates:
    try:
        DATA_ROOT = str(find_dataset_root(c))
        print("
resolved dataset root:", DATA_ROOT)
        break
    except FileNotFoundError:
        continue

assert DATA_ROOT, (
    "LIDC slices not found. Add the 'zhangweiled/lidcidri' dataset via "
    "Add data, then re-run this cell."
)

In [ ]:
# ── 3. Smoke test (~3 min) ────────────────────────────────
# Proves the whole path works before spending GPU hours. Run this first.
!python tests/test_no_tf.py
!python tests/test_models_tf.py
!python -m lidcseg.train --config configs/smoke.yaml --set data.root={DATA_ROOT}


## Index & split sanity check

Prints the dataset shape and confirms zero patient overlap. Worth a look
before a long run — it is also where you can see how many slices the original
`nodule-0`-only filter was discarding.

In [ ]:
from lidcseg.config import load_config
from lidcseg.data.index import build_index, index_stats
from lidcseg.data.splits import make_split
import json

cfg = load_config("configs/improved.yaml", [f"data.root={DATA_ROOT}"])

recs_all = build_index(cfg.data.root, None, cfg.data.n_readers)
recs_n0  = build_index(cfg.data.root, ["nodule-0"], cfg.data.n_readers)
print("all nodules :", json.dumps(index_stats(recs_all)))
print("nodule-0 only:", json.dumps(index_stats(recs_n0)))
print(f"
-> using all nodules adds {len(recs_all) - len(recs_n0)} slices "
      f"({100 * (len(recs_all) / max(len(recs_n0), 1) - 1):.1f}% more)")

for mode in ("patient", "slice"):
    s = make_split(recs_all, mode=mode, seed=cfg.split.seed)
    print(f"
{mode:8s} {json.dumps(s.summary(recs_all))}")

## Main run

`configs/improved.yaml` — patient split, all nodules, ImageNet-correct input
scaling, full LinkNet skips, per-image loss, fresh augmentation each epoch,
AdamW + cosine warmup, 8-way TTA.

Writes weights, `results.json`, `history.csv` and figures to
`/kaggle/working/runs/improved/`.

In [ ]:
!python -m lidcseg.train --config configs/improved.yaml --set data.root={DATA_ROOT}

## Baseline comparison

Two runs that make the leakage cost explicit:

- `notebook_best` — the original recipe *and* its slice-level split. Should
  land near the notebook's 0.8826 global Dice; it is the check that this
  refactor did not change the model.
- `notebook_best_patient` — same recipe, patient split. The gap between them
  is leakage, not modelling.

In [ ]:
!python -m lidcseg.train --config configs/notebook_best.yaml --set data.root={DATA_ROOT}
!python -m lidcseg.train --config configs/notebook_best_patient.yaml --set data.root={DATA_ROOT}

## Ablations

One change per variant, so each difference is attributable. The leaderboard is
rewritten after every variant, so an interrupted sweep still leaves results.

`ablation` is 11 runs — start it only when you have the GPU budget, or pass
`--only` to pick a few.

In [ ]:
!python -m lidcseg.experiments --config configs/improved.yaml --suite attention     --set data.root={DATA_ROOT}

In [ ]:
# Full ablation (long). Trim with --only no_attention no_aug global_loss
# !python -m lidcseg.experiments --config configs/improved.yaml --suite ablation #     --set data.root={DATA_ROOT}

# 5-fold patient-grouped CV — the number to quote in a write-up.
# !python -m lidcseg.experiments --config configs/improved.yaml --suite cv5 #     --set data.root={DATA_ROOT}

## Baselines — is this architecture the right choice?

The ablations compare the model against versions of itself, which cannot say
whether the architecture was worth picking. These two run through the identical
pipeline — same split, loss, augmentation, schedule, evaluation — so any
difference is architectural.

| arch | encoder | decoder | attention |
|---|---|---|---|
| `dplinknet` | ResNet50 (ImageNet) | LinkNet (additive skips) | SCANeXt |
| `resnet50_unet` | ResNet50 (ImageNet) | U-Net (concat skips) | none |
| `unet` | random init | U-Net | none |

`improved − resnet50_unet` is what the LinkNet decoder plus attention buys.
`resnet50_unet − unet` is what the ImageNet encoder buys.

They are **not parameter-matched** (~80M / ~60M / ~31M), so read the params
column in the leaderboard. For a parameter-matched attention comparison, use
the `none` row of `--suite attention` instead.

In [ ]:
# Both baselines plus all three attention settings, one leaderboard.
!python -m lidcseg.experiments --config configs/improved.yaml --suite baselines     --set data.root={DATA_ROOT}

In [ ]:
# Or run them individually.
# !python -m lidcseg.train --config configs/baseline_resnet50_unet.yaml --set data.root={DATA_ROOT}
# !python -m lidcseg.train --config configs/baseline_unet.yaml --set data.root={DATA_ROOT}

In [ ]:
# Show any leaderboard written so far.
import pathlib
from IPython.display import Markdown, display

for lb in sorted(pathlib.Path("/kaggle/working/runs").glob("*_leaderboard.md")):
    display(Markdown(lb.read_text()))

## Results

In [ ]:
import json, pathlib
from IPython.display import Image, display, Markdown

for run in sorted(pathlib.Path("/kaggle/working/runs").glob("*/")):
    res = run / "results.json"
    if not res.exists():
        continue
    r = json.loads(res.read_text())
    t = r["test"]
    ci = t.get("dice_ci")
    display(Markdown(
        f"### `{run.name}`
"
        f"- per-image Dice **{t['dice']:.4f}**"
        + (f" (95% CI {ci[0]:.4f}-{ci[1]:.4f})" if ci else "")
        + f"
- IoU {t['iou']:.4f} | global Dice {t['dice_global']:.4f}"
          f"
- precision {t['precision']:.4f} | recall {t['recall']:.4f}"
          f"
- threshold {r['threshold']:.3f} | TTA {r['tta']}"
    ))
    for fig in ("training_curves.png", "test_metrics.png", "predictions.png"):
        p = run / fig
        if p.exists():
            display(Image(str(p)))